# QLoRA Fine-Tuning on Colab

**Model:** Qwen/Qwen2.5-7B-Instruct  
**Method:** QLoRA (4-bit NF4 + LoRA)  
**GPU:** T4 (16GB VRAM) — Colab Free Tier  

Open in Colab: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghassan-gaidi/finetune-lab/blob/main/notebooks/lora/example_qlora_colab.ipynb)

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -qU \
    transformers \
    accelerate \
    peft \
    bitsandbytes \
    datasets \
    trl \
    huggingface_hub \
    wandb

## 2. Model & Tokenizer Setup

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

## 3. LoRA Config

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## 4. Load Dataset

In [ ]:
# Example: load a small instruction dataset
dataset = load_dataset("HuggingFaceH4/no_robots", split="train[:1000]")

def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"], tokenize=False
        )
    }

dataset = dataset.map(format_chat)

## 5. Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./outputs/qwen-7b-qlora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    save_steps=100,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    formatting_func=lambda x: x["text"],
    max_seq_length=2048,
)

trainer.train()

## 6. Merge & Save

In [ ]:
# Push adapter to HF Hub
trainer.model.push_to_hub("ghassan-gaidi/qwen-7b-qlora-adapter")
tokenizer.push_to_hub("ghassan-gaidi/qwen-7b-qlora-adapter")
print("Adapter saved to HF Hub!")